In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# Create pilot subset (100 images)


# Paths
BASE_DIR = Path("..").resolve()

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

HAM_IMG_DIR    = BASE_DIR / "data" / "ham10000" / "images"
ISIC_IMG_DIR   = BASE_DIR / "data" / "isic2018" / "images"
HAM_CSV        = BASE_DIR / "data" / "preprocessed_manifests" / "ham10000_preprocessed.csv"
ISIC_CSV       = BASE_DIR / "data" / "preprocessed_manifests" / "isic2018_preprocessed.csv"

RUN_NAME = "pilot_1260"
PILOT_DIR = DATA_DIR / RUN_NAME
PILOT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

# Target size
PILOT_SIZE_TOTAL = 1260
PILOT_SIZE_HAM = 630
PILOT_SIZE_ISIC = 630

print(f"BASE_DIR: {BASE_DIR}")
print(f"RUN_NAME: {RUN_NAME}")
print(f"PILOT_DIR: {PILOT_DIR}")


# ------------------------------------------------------------
# Load manifests

ham_df = pd.read_csv(HAM_CSV)

isic_df = pd.read_csv(ISIC_CSV)

ham_df["dataset"] = "HAM10000"
isic_df["dataset"] = "ISIC2018"

# ------------------------------------------------------------
# HAM sampling

ham_mel = ham_df[ham_df["label"] == 1]
ham_nonmel = ham_df[ham_df["label"] == 0]

ham_sample = pd.concat([
    ham_mel.sample(
        n=PILOT_SIZE_HAM // 2,
        random_state=RANDOM_STATE
    ),
    ham_nonmel.sample(
        n=PILOT_SIZE_HAM // 2,
        random_state=RANDOM_STATE
    )
]).reset_index(drop=True)

# ------------------------------------------------------------
# ISIC sampling
# ------------------------------------------------------------

very_tiny = isic_df[isic_df["mask_size_class"] == "very_tiny"]

tiny = isic_df[isic_df["mask_size_class"] == "tiny"]

small = isic_df[isic_df["mask_size_class"] == "small"]

normal = isic_df[isic_df["mask_size_class"] == "normal"]

border_touching = isic_df[isic_df["touches_any_border"] == True]

N_VERY_TINY = min(50, len(very_tiny))
N_TINY = min(50, len(tiny))
N_SMALL = min(100, len(small))
N_BORDER = min(50, len(border_touching))

already_targeted = N_VERY_TINY + N_TINY + N_SMALL + N_BORDER
N_NORMAL = PILOT_SIZE_ISIC - already_targeted

isic_parts = [
    very_tiny.sample(n=N_VERY_TINY, random_state=RANDOM_STATE),
    tiny.sample(n=N_TINY, random_state=RANDOM_STATE),
    small.sample(n=N_SMALL, random_state=RANDOM_STATE),
    normal.sample(n=N_NORMAL, random_state=RANDOM_STATE),
    border_touching.sample(n=N_BORDER, random_state=RANDOM_STATE),
]

isic_sample = (
    pd.concat(isic_parts)
    .drop_duplicates(subset=["image_id"])
    .reset_index(drop=True)
)

# If overlap caused fewer than 630 rows, top up from remaining ISIC records.
if len(isic_sample) < PILOT_SIZE_ISIC:
    n_missing = PILOT_SIZE_ISIC - len(isic_sample)

    remaining_isic = isic_df[
        ~isic_df["image_id"].isin(isic_sample["image_id"])
    ]

    topup = remaining_isic.sample(
        n=n_missing,
        random_state=RANDOM_STATE
    )

    isic_sample = (
        pd.concat([isic_sample, topup])
        .reset_index(drop=True)
    )

print("ISIC sample:", len(isic_sample))
display(isic_sample["mask_size_class"].value_counts())
display(isic_sample["touches_any_border"].value_counts())

# Merge and save
# ------------------------------------------------------------

pilot_df = pd.concat([
    ham_sample,
    isic_sample
]).reset_index(drop=True)

pilot_csv = PILOT_DIR / "pilot_subset_1260.csv"

pilot_df.to_csv(pilot_csv, index=False)

print("Saved:", pilot_csv)
print("Total images:", len(pilot_df))

display(pilot_df.groupby("dataset").size())

if "label" in pilot_df.columns:
    display(pilot_df.groupby(["dataset", "label"]).size())

if "mask_size_class" in pilot_df.columns:
    display(pilot_df.groupby(["dataset", "mask_size_class"]).size())

if "touches_any_border" in pilot_df.columns:
    display(pilot_df.groupby(["dataset", "touches_any_border"]).size())
